# A first read-only task: synthetic repeated measurements

What common value summarizes five invented measurements under independent Gaussian statistical errors? You will inspect the small input, use the calculation in `analyze.py` and explain its limits. This is a calculation exercise, not a scientific result.

Use a Jupyter application configured to use the project's prepared Python environment for this tutorial. Restart the kernel and run all cells from the beginning. Launch from the repository root or `examples/repeated_measurements/`. Leave the source, data and notebook file unchanged; close without saving execution outputs.

## Load the source and locate the input

This setup checks that the working directory is one of the two supported locations and loads `analyze.py` from this example. It avoids relying on a previous import. Check that the displayed source and data paths belong to your checkout. If setup fails, reopen from one of the named directories; if an import fails, check that the kernel uses the tutorial's Python environment.

In [ ]:
from pathlib import Path
import importlib.util
import sys

cwd = Path.cwd().resolve()
candidates = []
for repo_root in (cwd, cwd.parent.parent):
    example = repo_root / "examples" / "repeated_measurements"
    if (
        cwd in (repo_root, example)
        and (repo_root / "docs/tutorials/03_complete_a_first_read_only_task.md").is_file()
        and (example / "analyze.py").is_file()
        and (example / "data/measurements.csv").is_file()
    ):
        candidates.append(example)
if len(candidates) != 1:
    raise RuntimeError(
        "Launch this notebook from the repository root or "
        "examples/repeated_measurements/ in your checkout, then restart and run all."
    )
example_dir = candidates[0]
source_path = example_dir / "analyze.py"
data_path = example_dir / "data/measurements.csv"
spec = importlib.util.spec_from_file_location("tutorial_analysis", source_path)
analysis = importlib.util.module_from_spec(spec)
# Avoid creating a bytecode cache beside the read-only maintained source.
previous_bytecode_setting = sys.dont_write_bytecode
try:
    sys.dont_write_bytecode = True
    spec.loader.exec_module(analysis)
finally:
    sys.dont_write_bytecode = previous_bytecode_setting
print("Maintained source:", source_path)
print("Read-only input:", data_path)

## Inspect the input without changing it

Read the CSV as text so you can see its header and five measurement identifiers. Then use the script's `read_measurements` function to check and read it; the notebook does not need a second CSV parser. Each row gives a value and a positive one-standard-deviation uncertainty. The numbers were invented for teaching. If a file is missing or the input fails a check, inspect the input and working directory before proceeding. Leave the files unchanged.

In [ ]:
print(data_path.read_text(encoding="utf-8"))
rows = analysis.read_measurements(data_path)
print("Measurements read:", len(rows))

## Run the calculation

`analyze.py` contains the maintained calculation. The next cell calls its `summarize` function and displays the returned values with readable labels. It also displays chi-square per degree of freedom, just as the script does. Compare these values with the terminal route in Tutorial 3; the script prints six decimal places.

In [ ]:
import json

count, mean, standard_error, chi_square, dof = analysis.summarize(rows)
result = {
    "measurements": count,
    "weighted_mean": mean,
    "standard_error": standard_error,
    "chi_square": chi_square,
    "degrees_of_freedom": dof,
    "chi_square_per_dof": chi_square / dof,
}
print(json.dumps(result, indent=2))

## Interpret what you observed

The five measurements are assumed to estimate a single common value, with independent Gaussian statistical errors. The weighted mean summarizes them under those assumptions. The standard error follows from the stated uncertainty model. Chi-square measures residual variation relative to the stated uncertainties; chi-square per degree of freedom is not a goodness-of-fit probability or a scientific acceptance decision.

For the fixed CSV, the script prints five measurements, mean `10.060000`, standard error `0.089443`, chi-square `4.300000`, four degrees of freedom and ratio `1.075000`. Matching these values reproduces this small example; it does not establish that the model is appropriate for real data. If values differ at six-decimal precision, stop and record the source and input paths and the discrepancy before using the results.

**Checkpoint:** Which file contains the maintained calculation? Which assumptions would you need to challenge before applying the result to real measurements? Why does successful execution leave those scientific questions open?

## Try it: does row order matter?

Predict whether reversing these same rows should change the summary. The next cell reverses a copy of the rows in memory and calls the same function. It does not change the CSV or implement another estimator. The equality check concerns this fixed example; it is not a general proof about floating-point arithmetic or scientific correctness.

In [ ]:
reordered_rows = list(reversed(rows))
reordered_summary = analysis.summarize(reordered_rows)
assert reordered_summary == analysis.summarize(rows), "Record this discrepancy before proceeding."
print("For this input, reversing row order leaves the returned summary unchanged.")

## Restart, then return to your task

Restart the kernel and run all cells from the beginning to check that your observations do not depend on variables left over from earlier work. If you see an undefined name, do this before interpreting results. The supported working directories are the repository root and the example directory; the setup reports an error if you launch it elsewhere. The repository copy has empty outputs and null execution counts. Close without saving your local outputs into this file.

Write a short account outside the repository: name the input and source files, the values you observed, the assumptions, and what remains unproven. Ask a reviewer or colleague who did not produce the original account to challenge it; you decide whether it answers this synthetic question.

Return to [Tutorial 3](../../docs/tutorials/03_complete_a_first_read_only_task.md) for the terminal comparison and evidence discussion. The nearby [README](README.md) explains which files provide the calculation and input, and describes the example's limits; the [manual](../../docs/index.md) offers the next reading route.